In [1]:
import matplotlib.pyplot as plt
import multiprocessing as mp
##matplotlib.use('Agg')
import pandas as pd
from sklearn.externals import joblib
import scipy
from itertools import combinations
import numpy as np
import ants
from skimage import measure
import json
import numpy as np
from numpy.testing import assert_raises
from nilearn import datasets, surface, image
from nilearn.plotting import html_surface, html_connectome
from nilearn.plotting.js_plotting_utils import decode
from nilearn.datasets import fetch_surf_fsaverage
from nilearn._utils.exceptions import DimensionError
from matplotlib import gridspec
from nilearn import surface, plotting, datasets
import seaborn as sns
import os, sys
import glob
import json
import os
import copy
from statsmodels.regression.linear_model import OLS
from sklearn.linear_model import Lasso, logistic, LinearRegression, \
LogisticRegression, RidgeClassifier, RidgeClassifierCV
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import xgboost
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, roc_auc_score, average_precision_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelBinarizer, RobustScaler
from sklearn.feature_selection import SelectFromModel, f_regression, SelectKBest, SelectPercentile, SelectFwe, \
  f_classif, SelectFdr
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, LeaveOneOut, KFold, RepeatedStratifiedKFold, RepeatedKFold
from scipy import stats
from shutil import copyfile
import numpy as np
import pandas as pd
import nibabel as nb

import xgboost as xgb
from imblearn.over_sampling import ADASYN, SMOTE, RandomOverSampler
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics.scorer import make_scorer
sns.set("paper", font_scale=2)
sns.set_style('ticks', {'font.family':'sans-serif', 'font.serif':'Helvetica'})
%matplotlib inline
%reload_ext rpy2.ipython
%lsmagic

/home/qinqian/.local/lib/python3.6/site-packages/sklearn/externals/joblib/__init__.py:15: DeprecationWarning: sklearn.externals.joblib is deprecated in 0.21 and will be removed in 0.23. Please import this functionality directly from joblib, which can be installed with: pip install joblib. If this warning is raised when loading pickled models, you may need to re-serialize those models with scikit-learn 0.21+.
  warnings.warn(msg, category=DeprecationWarning)
Using TensorFlow backend.


Limited tf.compat.v2.summary API due to missing TensorBoard installation
Limited tf.summary API due to missing TensorBoard installation


Available line magics:
%R  %Rdevice  %Rget  %Rpull  %Rpush  %alias  %alias_magic  %autocall  %automagic  %autosave  %bookmark  %cat  %cd  %clear  %colors  %config  %connect_info  %cp  %debug  %dhist  %dirs  %doctest_mode  %ed  %edit  %env  %gui  %hist  %history  %killbgscripts  %ldir  %less  %lf  %lk  %ll  %load  %load_ext  %loadpy  %logoff  %logon  %logstart  %logstate  %logstop  %ls  %lsmagic  %lx  %macro  %magic  %man  %matplotlib  %mkdir  %more  %mv  %notebook  %page  %pastebin  %pdb  %pdef  %pdoc  %pfile  %pinfo  %pinfo2  %popd  %pprint  %precision  %profile  %prun  %psearch  %psource  %pushd  %pwd  %pycat  %pylab  %qtconsole  %quickref  %recall  %rehashx  %reload_ext  %rep  %rerun  %reset  %reset_selective  %rm  %rmdir  %run  %save  %sc  %set_env  %store  %sx  %system  %tb  %time  %timeit  %unalias  %unload_ext  %who  %who_ls  %whos  %xdel  %xmode

Available cell magics:
%%!  %%HTML  %%R  %%SVG  %%bash  %%capture  %%debug  %%file  %%html  %%javascript  %%js  %%latex  %%markdown  

In [3]:
jhu_df = pd.read_csv('jhu_roi_model_raw_fillna.csv')
peaktsb = pd.read_csv('peak_tsb.txt', sep='\t')
final = jhu_df.merge(peaktsb, left_on='participant_id', right_on='participant_id')

outcomes_continue = ['Cognitive_12m', 'Fine Motor_12m',
       'Gross Motor_12m', 'Receptive Communication_12m',
       'Expressive Communication_12m', 'Cognitive_18m', 'Fine Motor_18m',
       'Gross Motor_18m', 'Receptive Communication_18m',
       'Expressive Communication_18m']

response = final.loc[:, outcomes_continue]
brain_markers = final.loc[:, list(final.columns[1:(122*9+1)])+['BEAP', 'Peak-TSB', 'GA', 'CGA', 'bw', 'sex']]

In [4]:
tsb_group = final.loc[:, outcomes_continue]
tsb_group.loc[:, 'Peak_TSB'] = final.loc[:, 'Peak-TSB']
tsb_group.loc[:, 'GA'] = final.loc[:, 'GA']
tsb_group.loc[:, 'CGA'] = final.loc[:, 'CGA']
tsb_group.loc[:, 'BW'] = final.loc[:, 'bw']
#tsb_group1 = tsb_group.loc[tsb_group.GA_Group == 1, :]
#tsb_group2 = tsb_group.loc[tsb_group.GA_Group == 0, :]
#tsb_group1.drop(['GA_Group'])
tsb_group = tsb_group.melt(id_vars=['Peak_TSB', 'CGA', 'GA', 'BW'], value_name='Follow up')
%Rpush tsb_group
%R colnames(tsb_group)[5] = 'Outcome'
%R tsb_group$Time = factor(apply(matrix(tsb_group$Outcome), 1, function(x) unlist(strsplit(x, '_'))[2]))
%R tail(tsb_group)

,Peak_TSB,CGA,GA,BW,Outcome,Follow.up,Time
1734,25.5,39.861429,38.29,3500.0,Expressive Communication_18m,92,18m
1735,22.1,35.140000,32.14,1550.0,Expressive Communication_18m,134,18m
1736,32.4,32.852857,32.71,2030.0,Expressive Communication_18m,97,18m
1737,27.8,53.000000,41.00,3300.0,Expressive Communication_18m,126,18m
1738,33.8,33.710000,29.71,1500.0,Expressive Communication_18m,138,18m
1739,27.0,49.430000,37.43,2910.0,Expressive Communication_18m,119,18m


In [5]:
%%R
library(ggplot2)
library(ggpubr)
print(head(tsb_group))
#print(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))))
ggscatter(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))),
          x = "Peak_TSB", y = "Follow.up", add = "reg.line",
          conf.int = TRUE) +
  facet_wrap(~Time+Outcome, ncol=5) +
  stat_cor(label.x = 20, label.y = 152, method='pearson') +
  stat_regline_equation(label.x = 20, label.y = 160)
#p = ggboxplot(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))), 
#              x='TSB_Group',
#              y='Follow.up', palette = "npg",
#              color='TSB_Group', short.panel.labs = FALSE,
#              add="jitter") + facet_wrap(~ Time + Outcome, ncol=5)
#p + stat_compare_means(method='t.test')
ggsave("Fig2_TSB_linear.pdf", width=13, height=8)

W0718 08:54:16.127277 140471820175104 callbacks.py:116] R[write to console]: Loading required package: magrittr



  Peak_TSB      CGA    GA   BW       Outcome Follow.up Time
0     15.3 41.14000 38.14 3680 Cognitive_12m        96  12m
1     22.1 34.99714 34.14 2265 Cognitive_12m       104  12m
2     30.1 38.29000 30.29  940 Cognitive_12m       122  12m
3     15.4 35.00000 34.00 2220 Cognitive_12m       108  12m
4     19.3 34.43143 32.86 2380 Cognitive_12m        77  12m
5     21.5 35.14286 33.00 2050 Cognitive_12m        77  12m


In [6]:
%%R
library(ggplot2)
library(ggpubr)
print(head(tsb_group))
#print(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))))
ggscatter(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))),
          x = "GA", y = "Follow.up", add = "reg.line",
          conf.int = TRUE) +
  facet_wrap(~Time+Outcome, ncol=5) +
  stat_cor(label.x = 31, label.y = 152, method='pearson') +
  stat_regline_equation(label.x = 31, label.y = 160)
#p = ggboxplot(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))), 
#              x='TSB_Group',
#              y='Follow.up', palette = "npg",
#              color='TSB_Group', short.panel.labs = FALSE,
#              add="jitter") + facet_wrap(~ Time + Outcome, ncol=5)
#p + stat_compare_means(method='t.test')
ggsave("Fig2_GA_linear.pdf", width=13, height=8)

  Peak_TSB      CGA    GA   BW       Outcome Follow.up Time
0     15.3 41.14000 38.14 3680 Cognitive_12m        96  12m
1     22.1 34.99714 34.14 2265 Cognitive_12m       104  12m
2     30.1 38.29000 30.29  940 Cognitive_12m       122  12m
3     15.4 35.00000 34.00 2220 Cognitive_12m       108  12m
4     19.3 34.43143 32.86 2380 Cognitive_12m        77  12m
5     21.5 35.14286 33.00 2050 Cognitive_12m        77  12m


In [7]:
%%R
library(ggplot2)
library(ggpubr)
print(head(tsb_group))
#print(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))))
ggscatter(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))),
          x = "CGA", y = "Follow.up", add = "reg.line",
          conf.int = TRUE) +
  facet_wrap(~Time+Outcome, ncol=5) +
  stat_cor(label.x = 31, label.y = 152, method='pearson') +
  stat_regline_equation(label.x = 31, label.y = 160)
#p = ggboxplot(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))), 
#              x='TSB_Group',
#              y='Follow.up', palette = "npg",
#              color='TSB_Group', short.panel.labs = FALSE,
#              add="jitter") + facet_wrap(~ Time + Outcome, ncol=5)
#p + stat_compare_means(method='t.test')
ggsave("Fig2_CGA_linear.pdf", width=13, height=8)

  Peak_TSB      CGA    GA   BW       Outcome Follow.up Time
0     15.3 41.14000 38.14 3680 Cognitive_12m        96  12m
1     22.1 34.99714 34.14 2265 Cognitive_12m       104  12m
2     30.1 38.29000 30.29  940 Cognitive_12m       122  12m
3     15.4 35.00000 34.00 2220 Cognitive_12m       108  12m
4     19.3 34.43143 32.86 2380 Cognitive_12m        77  12m
5     21.5 35.14286 33.00 2050 Cognitive_12m        77  12m


In [8]:
%%R
library(ggplot2)
library(ggpubr)
print(head(tsb_group))
#print(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))))
ggscatter(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))),
          x = "BW", y = "Follow.up", add = "reg.line",
          conf.int = TRUE) +
  facet_wrap(~Time+Outcome, ncol=5) +
  stat_cor(label.x = 31, label.y = 152, method='pearson') +
  stat_regline_equation(label.x = 31, label.y = 160)
#p = ggboxplot(subset(tsb_group, (!(Outcome %in% c("CGA", "BW", "GA", "Peak_TSB")))), 
#              x='TSB_Group',
#              y='Follow.up', palette = "npg",
#              color='TSB_Group', short.panel.labs = FALSE,
#              add="jitter") + facet_wrap(~ Time + Outcome, ncol=5)
#p + stat_compare_means(method='t.test')
ggsave("Fig2_BW_linear.pdf", width=13, height=8)

  Peak_TSB      CGA    GA   BW       Outcome Follow.up Time
0     15.3 41.14000 38.14 3680 Cognitive_12m        96  12m
1     22.1 34.99714 34.14 2265 Cognitive_12m       104  12m
2     30.1 38.29000 30.29  940 Cognitive_12m       122  12m
3     15.4 35.00000 34.00 2220 Cognitive_12m       108  12m
4     19.3 34.43143 32.86 2380 Cognitive_12m        77  12m
5     21.5 35.14286 33.00 2050 Cognitive_12m        77  12m


In [10]:
## https://zhuanlan.zhihu.com/p/27491381
baep_analysis = final.loc[:, ['BEAP', 'GA', 'CGA','Peak-TSB']+outcomes_continue]
baep_analysis.loc[:, 'TSB_Group'] = 'Medium'
baep_analysis.loc[~(baep_analysis.loc[:, 'Peak-TSB'] >= 15) & (baep_analysis.loc[:, 'Peak-TSB'] < 20), 'TSB_Group'] = 'Severe'
baep_analysis.loc[:, 'GA_Group'] = 'Preterm'
baep_analysis.loc[final.GA > 35, 'GA_Group'] = '(Near) Term'
baep_analysis = baep_analysis.loc[~pd.isnull(baep_analysis.BEAP), ]
baep_analysis_melt = baep_analysis.melt(id_vars=['BEAP', 'TSB_Group', 'GA_Group'], value_name='Follow up')
print(baep_analysis_melt.head())

%Rpush baep_analysis_melt
%R baep_analysis_melt$BEAP = as.factor(baep_analysis_melt$BEAP)
%R colnames(baep_analysis_melt)[4] = 'Outcome'
%R baep_analysis_melt$Time = factor(apply(matrix(baep_analysis_melt$Outcome), 1, function(x) unlist(strsplit(x, '_'))[2]))                                          
%R baep_analysis_melt$Time = factor(baep_analysis_melt$Time, levels=unique(baep_analysis_melt$Time))

   BEAP TSB_Group GA_Group variable  Follow up
0   0.0    Medium  Preterm       GA      34.14
1   1.0    Medium  Preterm       GA      30.29
2   0.0    Medium  Preterm       GA      32.86
3   0.0    Medium  Preterm       GA      33.00
4   0.0    Medium  Preterm       GA      34.29


[NaN, NaN, NaN, NaN, NaN, ..., 18m, 18m, 18m, 18m, 18m]
Length: 1417
Categories (2, object): , 18m]

In [11]:
%%R
library(ggplot2)
library(ggpubr)
p = ggboxplot(subset(baep_analysis_melt, !(Outcome%in% c("GA", "CGA", "Peak-TSB"))), 
              x='BEAP',
              y='Follow.up', palette = "npg",
              color='BEAP', short.panel.labs = FALSE,
              add="jitter") + facet_wrap( ~ Time + Outcome, ncol=5)
p + stat_compare_means(method='t.test') + ylim(c(0,150))
ggsave("Fig2_BEAP.pdf", width=12, height=8)